# Lección 8: Instrumentos Ópticos y Ecuaciones de Fresnel
### Fundamentos de Óptica y Oscilaciones

---

Los instrumentos ópticos (como telescopios y microscopios) combinan elementos refractores y reflectores para manipular el camino óptico de la luz, permitiendo la magnificación y observación detallada del universo macroscópico y microscópico. La eficiencia de estos sistemas depende del control de las reflexiones en las interfaces, gobernadas por las ecuaciones de Fresnel, y de la mitigación de las aberraciones ópticas. En esta lección se formalizará el estudio de los instrumentos ópticos, derivando los coeficientes de reflexión de Fresnel y el invariante de Abbe para dioptrios esféricos. Asimismo, se analizará la física de las aberraciones y su corrección mediante dobletes acromáticos, y se modelará computacionalmente la propagación de rayos en telescopios (galileano y kepleriano) y microscopios compuestos.

---


## Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Calcular** los coeficientes de reflexión y transmisión de Fresnel ($R_s, R_p, T_s, T_p$) y determinar el ángulo de Brewster para interfaces dieléctricas.
2. **Resolver** la refracción en superficies esféricas individuales (dioptrios) aplicando el invariante de Abbe.
3. **Analizar** el origen físico de las aberraciones cromática y esférica y su corrección mediante dobletes acromáticos.
4. **Modelar** y comparar el trazado de rayos en telescopios refractores de Galileo y Kepler, y caracterizar los telescopios reflectores (Newton, Cassegrain).
5. **Diferenciar** el funcionamiento y la magnificación de microscopios simples y compuestos.
6. **Validar** analíticamente las propiedades físicas de los sistemas de lentes compuestas y el ángulo de Brewster mediante aserciones numéricas en Python.


## 1. Espejos y Coeficientes de Fresnel

La cantidad de luz que se refleja y transmite al incidir sobre una interfaz entre dos medios dieléctricos depende de los índices de refracción, del ángulo de incidencia y del estado de polarización de la onda. Este comportamiento está gobernado por las **ecuaciones de Fresnel**.

### 1.1 Coeficientes de Reflexión y Transmisión
Dividimos la luz en dos componentes de polarización lineal respecto al plano de incidencia (plano que contiene al rayo incidente y a la normal de la superficie):
* **Polarización s (senkrecht):** El campo eléctrico de la onda es perpendicular al plano de incidencia.
* **Polarización p (paralela):** El campo eléctrico es paralelo al plano de incidencia.

Los coeficientes de reflectancia de potencia ($R_s$ y $R_p$), que indican la fracción de energía reflejada, vienen dados por:
$$R_s = \left| \frac{n_1 \cos\theta_i - n_2 \cos\theta_t}{n_1 \cos\theta_i + n_2 \cos\theta_t} \right|^2$$
$$R_p = \left| \frac{n_2 \cos\theta_i - n_1 \cos\theta_t}{n_2 \cos\theta_i + n_1 \cos\theta_t} \right|^2$$
Donde $\theta_i$ es el ángulo de incidencia y $\theta_t$ es el de refracción, relacionados por la ley de Snell ($\sin\theta_t = \frac{n_1}{n_2} \sin\theta_i$). Por conservación de la energía, las transmitancias son:
$$T_s = 1 - R_s, \quad T_p = 1 - R_p$$

### 1.2 Casos Especiales y Ángulo de Brewster
1. **Incidencia Normal ($\theta_i = 0$):** En este límite, la reflectancia es idéntica para ambas polarizaciones y se reduce a:
  $$R = \left( \frac{n_1 - n_2}{n_1 + n_2} \right)^2$$
2. **Ángulo de Brewster ($\theta_B$):** Es el ángulo de incidencia para el cual la reflectancia de la polarización p es nula ($R_p = 0$). En este punto, toda la componente paralela se transmite y la luz reflejada es $100\%$ polarizada de forma s. Se calcula como:
  $$\theta_B = \arctan\left( \frac{n_2}{n_1} \right)$$
3. **Reflexión Total Interna:** Si la luz viaja de un medio más denso a uno menos denso ($n_1 > n_2$), para ángulos superiores al crítico ($\theta_i > \theta_c = \arcsin(n_2/n_1)$), se tiene $R_s = R_p = 1.0$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Coeficientes de Fresnel para vidrio Flint (n1=1.5) a aire (n2=1.0)
n1 = 1.5
n2 = 1.0
theta_i_deg = np.linspace(0, 90, 500)
theta_i_rad = np.radians(theta_i_deg)

theta_c_rad = np.arcsin(n2 / n1)
theta_B_rad = np.arctan(n2 / n1)

Rs = []
Rp = []

for th_i in theta_i_rad:
    if th_i > theta_c_rad:
        Rs.append(1.0)
        Rp.append(1.0)
    else:
        sin_th_t = (n1 * np.sin(th_i)) / n2
        th_t = np.arcsin(sin_th_t)
        
        # Coeficientes s y p
        rs_num = n1 * np.cos(th_i) - n2 * np.cos(th_t)
        rs_den = n1 * np.cos(th_i) + n2 * np.cos(th_t)
        rp_num = n2 * np.cos(th_i) - n1 * np.cos(th_t)
        rp_den = n2 * np.cos(th_i) + n1 * np.cos(th_t)
        
        Rs.append((rs_num / rs_den)**2)
        Rp.append((rp_num / rp_den)**2)

Rs = np.array(Rs)
Rp = np.array(Rp)

plt.figure(figsize=(10, 6))
plt.plot(theta_i_deg, Rs, 'b-', linewidth=2.5, label='Polarización s ($R_s$)')
plt.plot(theta_i_deg, Rp, 'r-', linewidth=2.5, label='Polarización p ($R_p$)')
plt.axvline(np.degrees(theta_B_rad), color='orange', linestyle='--', label=f'Ángulo de Brewster $\\theta_B \\approx {np.degrees(theta_B_rad):.1f}^\\circ$')
plt.axvline(np.degrees(theta_c_rad), color='darkviolet', linestyle=':', label=f'Ángulo Crítico $\\theta_c \\approx {np.degrees(theta_c_rad):.1f}^\\circ$')

plt.title('Coeficientes de Reflectancia de Fresnel (Vidrio $n=1.5$ a Aire $n=1.0$)', fontsize=13, fontweight='bold')
plt.xlabel('Ángulo de Incidencia $\\theta_i$ (grados)', fontsize=11)
plt.ylabel('Reflectancia de Potencia $R$', fontsize=11)
plt.xlim(0, 90)
plt.ylim(-0.05, 1.05)
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(frameon=True, loc='upper left')
plt.tight_layout()
plt.show()


## 2. Dióptricos Ópticos e Invariante de Abbe

Un **dioptrio esférico** es un sistema óptico formado por una sola superficie esférica refractante que separa dos medios de índices de refracción distintos $n_1$ y $n_2$.

### 2.1 El Invariante de Abbe
Para rayos paraxiales (rayos cercanos al eje principal con ángulos pequeños), la refracción en el dioptrio cumple con el **invariante de Abbe** o relación de invariación:
$$n_1 \left( \frac{1}{r} - \frac{1}{s_1} \right) = n_2 \left( \frac{1}{r} - \frac{1}{s_2} \right)$$
Donde:
* $r$ es el radio de curvatura de la superficie.
* $s_1$ es la distancia desde el vértice del dioptrio al objeto.
* $s_2$ es la distancia desde el vértice a la imagen.

Despejando $s_2$ (distancia de la imagen):
$$\frac{n_2}{s_2} = \frac{n_2 - n_1}{r} + \frac{n_1}{s_1} \implies s_2 = \frac{n_2}{\frac{n_2 - n_1}{r} + \frac{n_1}{s_1}}$$


In [ ]:
# Calcular y graficar la distancia de la imagen s2 en función de s1
# para un dioptrio convexo con r = 10 cm, n1 = 1.0 (aire), n2 = 1.5 (vidrio)

r_diop = 10.0  # cm
n1_val = 1.0
n2_val = 1.5

s1_range = np.linspace(5.0, 50.0, 400)
# Ecuación: n2/s2 = (n2-n1)/r + n1/s1
power_diop = (n2_val - n1_val) / r_diop
s2_range = n2_val / (power_diop + n1_val / s1_range)

plt.figure(figsize=(10, 5))
plt.plot(s1_range, s2_range, 'g-', linewidth=2.5, label='Posición de Imagen $s_2$')
plt.axhline(n2_val / power_diop, color='red', linestyle='--', label=f'Foco del Dioptrio $f_2 = {n2_val/power_diop:.1f}$ cm')
plt.title('Refracción en un Dioptrio Esférico Convexo', fontsize=13, fontweight='bold')
plt.xlabel('Distancia del Objeto $s_1$ (cm)', fontsize=11)
plt.ylabel('Distancia de la Imagen $s_2$ (cm)', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(frameon=True, loc='lower right')
plt.tight_layout()
plt.show()


## 3. Lentes Delgadas y Aberraciones Ópticas

En sistemas ópticos reales, la aproximación de lentes delgadas presenta limitaciones llamadas **aberraciones**, que degradan la nitidez de la imagen:

1. **Aberración Cromática:** Debido a la dispersión cromática, el índice de refracción del material depende de la longitud de onda de la luz. Como consecuencia, las longitudes de onda cortas (azul) se refractan más y se enfocan más cerca de la lente que las largas (rojo).
2. **Aberración Esférica:** Los rayos paralelos que inciden en las zonas marginales (periféricas) de una lente esférica se refractan con mayor fuerza y se enfocan más cerca de la lente que los rayos paraxiales (centrales). Esto genera un plano focal difuso.

### 3.1 Corrección: El Doblete Acromático
Isaac Newton creyó que la aberración cromática era imposible de corregir. Sin embargo, en el siglo XVIII se patentó el **doblete acromático**: un sistema compuesto por dos lentes delgadas pegadas:
* Una lente positiva convergente hecha de **vidrio crown** (bajo índice, baja dispersión).
* Una lente negativa divergente hecha de **vidrio flint** (alto índice, alta dispersión).

El diseño compensa exactamente la dispersión cromática del rojo y del azul, logrando un foco común para ambas bandas espectrales.


In [ ]:
# Simulación geométrica de la aberración esférica en una lente simple
# Los rayos a mayor altura y se enfocan antes en el eje principal x

y_marginal = np.linspace(-2.0, 2.0, 9)
x_lens = 0.0
f0_paraxial = 10.0  # Foco paraxial (cm)
spherical_coef = 0.4  # Coeficiente de aberración

plt.figure(figsize=(12, 6))

# Lente
plt.axvline(x_lens, color='blue', linewidth=4, label='Lente Simple con Aberración')
plt.axhline(0, color='black', linewidth=1, linestyle='--')

for y in y_marginal:
    if y == 0:
        continue
    # Foco del rayo según su altura y: f(y) = f0 - coef * y^2
    f_y = f0_paraxial - spherical_coef * y**2
    
    # Trazar rayo incidente
    plt.plot([-5, 0], [y, y], 'g-', alpha=0.5)
    # Trazar rayo refractado pasando por f_y
    plt.plot([0, f_y], [y, 0], 'r-', alpha=0.7)
    # Extensión del rayo tras pasar por el foco
    plt.plot([f_y, 12], [0, -y * (12 - f_y) / f_y], 'r-', alpha=0.3)
    
plt.scatter([f0_paraxial], [0], color='darkred', s=100, zorder=5, label='Foco Paraxial')
plt.title('Cáustica de Enfoque por Aberración Esférica en Lente Simple', fontsize=13, fontweight='bold')
plt.xlabel('Posición en el eje principal X (cm)', fontsize=11)
plt.ylabel('Altura Y (cm)', fontsize=11)
plt.xlim(-5, 12)
plt.ylim(-3, 3)
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(frameon=True, loc='upper left')
plt.tight_layout()
plt.show()


## 4. Telescopios Ópticos

Los telescopios recogen la luz paralela de objetos extremadamente lejanos (estrellas) para formar una imagen magnificada.

### 4.1 Telescopios Refractores (Lentes)
Utilizan lentes delgadas para captar y enfocar la luz:
* **Telescopio Galileano:** Combina una lente convergente como objetivo ($f_1 > 0$) y una divergente como ocular ($f_2 < 0$).
  - *Distancia:* $d = f_1 + f_2 = f_1 - |f_2|$. El ocular se sitúa antes del foco del objetivo.
  - *Imagen:* Virtual, derecha y magnificada. Posee un campo visual estrecho.
* **Telescopio Kepleriano:** Combina dos lentes convergentes: objetivo ($f_1 > 0$) y ocular ($f_2 > 0$).
  - *Distancia:* $d = f_1 + f_2$.
  - *Imagen:* Virtual, invertida y magnificada. Mayor campo visual que el galileano.

### 4.2 Telescopios Reflectores (Espejos)
Diseñados inicialmente por Isaac Newton para evitar la aberración cromática. Usan un espejo cóncavo primario para captar la luz:
* **Newtoniano:** El espejo primario cóncavo refleja la luz hacia un espejo plano secundario inclinado $45^\circ$, enviando la imagen al lateral del tubo.
* **Cassegrain:** El espejo primario posee un agujero en su centro. La luz es reflejada por un espejo secundario convexo hiperbólico a través del agujero hacia el ocular.
* **Schmidt-Cassegrain:** Incorpora una lente correctora delgada al inicio del tubo para eliminar la aberración esférica del espejo primario.


In [ ]:
# Comparación del trazado de rayos en telescopios de Galileo y Kepler

f1 = 30.0  # Focal del objetivo (cm)
f2_kep = 10.0  # Ocular Kepler (cm)
f2_gal = -10.0  # Ocular Galileo (cm)

d_kep = f1 + f2_kep  # 40 cm
d_gal = f1 + f2_gal  # 20 cm

y_in = np.array([-1.5, -0.5, 0.5, 1.5])
x_in = -5.0
x_obj = 0.0

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# 1. Telescopio Galileano
ax1.axvline(x_obj, color='blue', linewidth=3, label='Objetivo ($f_1 = 30$)')
ax1.axvline(d_gal, color='red', linewidth=2, label='Ocular Galileano ($f_2 = -10$)')
ax1.axhline(0, color='black', linewidth=1, linestyle='--')
for y in y_in:
    # Entrada
    ax1.plot([x_in, x_obj], [y, y], 'orange')
    # Hacia el ocular (convergencia a x=30)
    y_hit_gal = y * (1.0 - d_gal / f1)
    ax1.plot([x_obj, d_gal], [y, y_hit_gal], 'orange')
    # Salida (paralelos desfasados)
    ax1.plot([d_gal, 45], [y_hit_gal, y_hit_gal], 'orange', linewidth=2)
ax1.set_title('Telescopio Galileano: Rayos paralelos emergen paralelos y derechos (haz comprimido)', fontsize=12, fontweight='bold')
ax1.set_xlim(-7, 47)
ax1.set_ylim(-2.5, 2.5)
ax1.grid(True, linestyle=':', alpha=0.5)
ax1.legend(loc='upper right')

# 2. Telescopio Kepleriano
ax2.axvline(x_obj, color='blue', linewidth=3, label='Objetivo ($f_1 = 30$)')
ax2.axvline(d_kep, color='darkred', linewidth=3, label='Ocular Kepleriano ($f_2 = 10$)')
ax2.axhline(0, color='black', linewidth=1, linestyle='--')
ax2.scatter(f1, 0, color='green', s=80, label='Foco común')
for y in y_in:
    # Entrada
    ax2.plot([x_in, x_obj], [y, y], 'orange')
    # Cruce por foco común a x=40
    y_hit_kep = -y * (f2_kep / f1)
    ax2.plot([x_obj, f1], [y, 0], 'orange')
    ax2.plot([f1, d_kep], [0, y_hit_kep], 'orange')
    # Salida (paralelos invertidos)
    ax2.plot([d_kep, 55], [y_hit_kep, y_hit_kep], 'orange', linewidth=2)
ax2.set_title('Telescopio Kepleriano: Rayos cruzan el foco y emergen paralelos e invertidos', fontsize=12, fontweight='bold')
ax2.set_xlim(-7, 57)
ax2.set_ylim(-2.5, 2.5)
ax2.grid(True, linestyle=':', alpha=0.5)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()


## 5. Microscopios Ópticos

A diferencia del telescopio, el microscopio observa objetos pequeños colocados a muy corta distancia de la lente.

### 5.1 Microscopio Simple
Consiste en una única lente convergente (lupa). El objeto se coloca dentro de la distancia focal ($d_o < f$). Esto forma una **imagen virtual, derecha y magnificada**.

### 5.2 Microscopio Compuesto
Combina dos sistemas de lentes convergentes para obtener aumentos mucho mayores:
1. **Objetivo:** Lente de distancia focal corta ($f_{\text{obj}}$) colocada muy cerca del objeto (justo fuera de su foco, $d_o > f_{\text{obj}}$). Produce una **imagen real, invertida y magnificada** dentro del tubo del microscopio.
2. **Ocular:** Lente colocada en el extremo opuesto del tubo. El ocular actúa como una lupa simple: se sitúa de tal manera que la imagen real intermedia cae exactamente en su plano focal, proyectando una **imagen virtual final altamente ampliada e invertida** al infinito para la cómoda observación del ojo.

La magnificación total $M$ es el producto del aumento lateral del objetivo y el aumento angular del ocular:
$$M = M_{\text{obj}} \times M_{\text{ocular}} = \left( -\frac{L}{f_{\text{obj}}} \right) \left( \frac{25\text{ cm}}{f_{\text{ocular}}} \right)$$
Donde $L$ es la longitud óptica del tubo y $25\text{ cm}$ es la distancia estándar de punto cercano del ojo humano.


In [ ]:
# Simulación de trazado de rayos en un microscopio compuesto
f_obj = 2.0  # cm
f_oc = 4.0   # cm
do_obj = 2.2 # cm (justo fuera del foco de 2 cm)
ho_obj = 0.3 # cm

# 1. Imagen intermedia del objetivo
di_obj = (f_obj * do_obj) / (do_obj - f_obj)  # 22.0 cm
m_obj = -di_obj / do_obj  # -10
hi_obj = m_obj * ho_obj  # -3.0 cm

# Colocamos el ocular de modo que su foco coincida con la imagen intermedia en x = 22 cm
x_obj_lens = 0.0
x_intermediate_image = di_obj  # 22.0 cm
x_oc_lens = x_intermediate_image + f_oc  # 26.0 cm

plt.figure(figsize=(14, 6))
plt.axhline(0, color='black', linewidth=1, linestyle='--')

# Lentes
plt.axvline(x_obj_lens, color='blue', linewidth=3, label='Lente Objetivo ($f_{obj}=2$)')
plt.axvline(x_oc_lens, color='darkblue', linewidth=3, label='Lente Ocular ($f_{oc}=4$)')

# Objeto (azul)
plt.arrow(-do_obj, 0, 0, ho_obj, width=0.15, head_width=0.4, head_length=0.1, fc='blue', ec='blue', zorder=5, label='Objeto')

# Imagen intermedia (roja invertida)
plt.arrow(x_intermediate_image, 0, 0, hi_obj, width=0.15, head_width=0.4, head_length=0.1, fc='red', ec='red', zorder=5, label='Imagen Intermedia')

# Rayos del objetivo
# Rayo paralelo -> foco objetivo (x=2)
plt.plot([-do_obj, 0], [ho_obj, ho_obj], 'orange')
plt.plot([0, x_intermediate_image], [ho_obj, hi_obj], 'orange')
# Rayo central -> sin desvío
plt.plot([-do_obj, x_intermediate_image], [ho_obj, hi_obj], 'purple')

# Rayos del ocular saliendo paralelos al infinito
# Los rayos parten del extremo de la imagen intermedia (22, -3) hacia el ocular en x=26
# e indudablemente emergen paralelos con un ángulo dado por arctan(hi_obj/f_oc)
plt.plot([x_intermediate_image, x_oc_lens], [hi_obj, hi_obj], 'orange')
plt.plot([x_oc_lens, 32], [hi_obj, hi_obj - 6.0 * (hi_obj/f_oc)], 'orange', linewidth=2, label='Rayos paralelos al Ojo')
plt.plot([x_intermediate_image, x_oc_lens], [hi_obj, 0], 'purple')
plt.plot([x_oc_lens, 32], [0, - 6.0 * (hi_obj/f_oc)], 'purple', linewidth=2)

plt.title('Camino Óptico y Trazado de Rayos en un Microscopio Compuesto', fontsize=13, fontweight='bold')
plt.xlabel('Eje Óptico X (cm)', fontsize=11)
plt.ylabel('Altura Y (cm)', fontsize=11)
plt.xlim(-4, 34)
plt.ylim(-6, 6)
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(frameon=True, loc='upper left')
plt.tight_layout()
plt.show()


## 6. Verificación de Resultados Numéricos

Validamos la reflectividad de las ecuaciones de Fresnel y la posición analítica del ángulo de Brewster para interfaces vidrio-aire mediante la búsqueda numérica del cero de reflectancia.


In [ ]:
from scipy.optimize import minimize_scalar

n1_glass = 1.5
n2_air = 1.0

def Rp_function(theta_i):
    # Retornar Rp para optimizador
    sin_th_t = (n1_glass * np.sin(theta_i)) / n2_air
    if sin_th_t > 1.0:
        return 1.0  # TIR
    th_t = np.arcsin(sin_th_t)
    rp_num = n2_air * np.cos(theta_i) - n1_glass * np.cos(th_t)
    rp_den = n2_air * np.cos(theta_i) + n1_glass * np.cos(th_t)
    return (rp_num / rp_den)**2

# Encontramos el ángulo de Brewster de forma numérica
res_brewster = minimize_scalar(Rp_function, bounds=(0, np.radians(40.0)), method='bounded')
theta_B_num = np.degrees(res_brewster.x)
Rp_min_val = res_brewster.fun

# Ecuación teórica: tan(theta_B) = n2 / n1 => theta_B = arctan(n2/n1)
theta_B_theory = np.degrees(np.arctan(n2_air / n1_glass))

print('--- VERIFICACIÓN DEL ÁNGULO DE BREWSTER (VIDRIO -> AIRE) ---')
print(f'Ángulo de Brewster numérico (Mínimo Rp): {theta_B_num:.6f}°')
print(f'Valor mínimo de Rp: {Rp_min_val:.3e}')
print(f'Ángulo de Brewster teórico: {theta_B_theory:.6f}°')

assert np.isclose(theta_B_num, theta_B_theory, atol=1e-5)
print('\nValidación de Ecuaciones de Fresnel: ¡EXITOSA! (La búsqueda numérica y la fórmula coinciden perfectamente)')


## Bibliografía

[1] Gutiérrez, M. A.; Cáceres, W. A.; Pérez, J. “¿Qué es impedancia?”, Estudiantes curso de electromagnetismo, Universidad Autónoma de Bucaramanga, 2018. Disponible en: [Impedancia Blogspot](http://electromagnetismounab.blogspot.com/2018/04/que-es-impedancia.html).

[2] Nave, R.; Olmo, M; “Reflexión interna total”, *Hyperphysics*. Disponible en: [Hyperphysics](http://hyperphysics.phy-astr.gsu.edu/hbase/phyopt/totint.html).

[3] Hooke, R; Solís, C. (traductor), *Micrografía*, Editorial Alfaguara, 1989. Disponible en: [ResearchGate](https://www.researchgate.net/profile/Carlos-Solis-8/publication/307934900_Robert_Hooke_Micrografia/links/57d284d308ae0c0081e23f03/Robert-Hooke-Micrografia.pdf).

[4] Diccionario Collins, “Speculum metal”, *Collins English Dictionary*, 2010. Disponible en: [Collins](https://www.collinsdictionary.com/es/diccionario/ingles/speculum-metal).
